# filter() and Group Filtering

## Definition
`GroupBy.filter()` keeps or discards **entire groups** based on a condition applied to each group.
This is different from boolean row filtering — it operates at the group level.

### filter() vs other methods
| Method | Level | Returns |
|--------|-------|---------|
| `df[mask]` | Row | Rows where mask is True |
| `groupby().filter()` | Group | Rows of groups satisfying condition |
| `groupby().agg()` | Group → Scalar | One row per group |
| `groupby().transform()` | Row (group-aware) | Same shape as input |

### Use Cases
- Keep only groups with N+ members
- Keep only groups whose mean exceeds a threshold
- Remove minority groups from training data

In [ ]:
import pandas as pd
import numpy as np

df = pd.DataFrame({
    'dept':   ['Eng', 'Mkt', 'Eng', 'HR', 'Eng', 'Mkt', 'HR', 'Eng', 'Finance'],
    'salary': [90000, 65000, 75000, 85000, 95000, 80000, 70000, 70000, 110000],
    'score':  [88, 72, 91, 75, 84, 78, 69, 82, 95],
    'name':   ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I']
})
print(df.groupby('dept').size().to_frame('count'))

In [ ]:
# filter() — keep groups with at least 3 members
result = df.groupby('dept').filter(lambda grp: len(grp) >= 3)
print('Groups with >= 3 members:')
print(result)
print('Depts kept:', result['dept'].unique())

In [ ]:
# filter() — keep groups where mean salary > 75000
result2 = df.groupby('dept').filter(lambda grp: grp['salary'].mean() > 75000)
print('Groups with mean salary > 75000:')
print(result2[['dept', 'salary']].sort_values('dept'))

In [ ]:
# filter() — keep groups where ALL scores > 70
result3 = df.groupby('dept').filter(lambda grp: (grp['score'] > 70).all())
print('Groups where ALL scores > 70:')
print(result3[['dept', 'score']])

In [ ]:
# Combining filter with other operations
high_performing = (df
    .groupby('dept')
    .filter(lambda g: g['salary'].mean() > 75000 and len(g) >= 2)
)
print('High-performing large groups:')
print(high_performing)

In [ ]:
# Row-level filtering (traditional) vs group-level filter
# Row-level: keep individual rows where salary > 80000
row_filtered = df[df['salary'] > 80000]
print('Row filter (salary > 80k):')
print(row_filtered[['dept', 'name', 'salary']])

# Group-level: keep ALL rows of groups where MEAN salary > 80000
group_filtered = df.groupby('dept').filter(lambda g: g['salary'].mean() > 80000)
print('\nGroup filter (mean salary > 80k):')
print(group_filtered[['dept', 'name', 'salary']].sort_values('dept'))

In [ ]:
# ML use case: remove rare classes from training data
labels = pd.DataFrame({
    'label': ['cat', 'dog', 'cat', 'bird', 'dog', 'cat', 'fish', 'dog', 'cat', 'dog'],
    'value': range(10)
})
print('Class distribution:')
print(labels['label'].value_counts())

# Keep only classes with >= 3 samples
balanced = labels.groupby('label').filter(lambda g: len(g) >= 3)
print('\nAfter removing rare classes:')
print(balanced['label'].value_counts())